In [0]:
storage_account_name = "portfolioalba"
container_name = "raw-data"
access_key = dbutils.secrets.get(scope="portfolio-secrets", key="datalake-key")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", access_key)

file_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/clean_train.csv"

df_spark = spark.read.csv(file_path, header=True, inferSchema=True)

display(df_spark)


text,brand,sentiment
i have a 3g iphone after 3 hrs tweeting at riseaustin it was dead i need to upgrade plugin stations at sxsw,iPhone,Negative emotion
know about awesome ipadiphone app that youll likely appreciate for its design also theyre giving free ts at sxsw,iPad or iPhone App,Positive emotion
can not wait for ipad 2 also they should sale them down at sxsw,iPad,Positive emotion
i hope this years festival isnt as crashy as this years iphone app sxsw,iPad or iPhone App,Negative emotion
great stuff on fri sxsw marissa mayer google tim oreilly tech booksconferences amp matt mullenweg wordpress,Google,Positive emotion
new ipad apps for speechtherapy and communication are showcased at the sxsw conference iear edchat asd,null,No emotion toward brand or product
sxsw is just starting ctia is around the corner and googleio is only a hop skip and a jump from there good time to be an android fan,Android,Positive emotion
beautifully smart and simple idea rt wrote about our hollergram ipad app for sxsw,iPad or iPhone App,Positive emotion
counting down the days to sxsw plus strong canadian dollar means stock up on apple gear,Apple,Positive emotion
excited to meet the at sxsw so i can show them my sprint galaxy s still running android 21 fail,Android,Positive emotion


### Group by the “sentiment” column and count how many rows there are for each

In [0]:
from pyspark.sql.functions import count

record_df = df_spark.groupBy("sentiment").agg(count("*").alias("total_tweets"))

display(record_df)

sentiment,total_tweets
Negative emotion,519
null,164
No emotion toward brand or product,5388
Positive emotion,2672


### Saving the DataFrame in Parquet format

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/processed_data/sentiment_record"

record_df.write.mode("overwrite").parquet(output_path)

print(f"Data written to {output_path}")
                                          

Data written to abfss://raw-data@portfolioalba.dfs.core.windows.net/processed_data/sentiment_record


### Sending data to Azure SQL Database

In [0]:
server_name = "sentiments-server.database.windows.net"
database_name = "sentiments-db"
username = "CloudSAb35b1b88"
password = dbutils.secrets.get(scope="portfolio-secrets", key="sql-password")

jdbc_url = f"jdbc:sqlserver://{server_name}:1433;database={database_name}"

connection_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

record_df.write.jdbc(
    url=jdbc_url, 
    table="SentimentsRecord", 
    mode="overwrite", 
    properties=connection_properties
)

print("Data sent to Azure SQL Database successfully")


Data sent to Azure SQL Database successfully
